# Chapter 1: Fundamentals of AI Agents

Estimated time: about 6 hours. Roughly 30-45 minutes of that is one-time account/API setup,
longer than usual on purpose, since it is written for a reader with limited Python
background; the remaining 5+ hours is concept, build, break-it, and the interview drill.

Prerequisites: none beyond the repo-level Prerequisites gut-check in the root
`README.md`. This chapter assumes zero prior agent experience and does not assume you are
comfortable with a terminal, `pip`, or environment variables yet.

Interview category this chapter maps to: the baseline vocabulary and judgment questions
almost every AI-agent-engineering interview opens with. "What is an agent, really?" ReAct
basics. "Your agent is stuck in a loop, what do you do?"

## Setup

A small bit of bootstrapping so this notebook works the same way whether you open it from
the repo root or from inside `curriculum/` (Jupyter's working directory varies depending on
how you launch it), plus a fixed random seed: every notebook in this course seeds its
randomness so outputs are reproducible across machines and CI runs.

In [1]:
import sys
from pathlib import Path

# Make `agentlib` importable regardless of this notebook's working directory.
_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import json
import random
import re

from agentlib.grading import check

random.seed(42)
print(f"Repo root on sys.path: {_repo_root}")


Repo root on sys.path: /home/user/learning-agentic-ai


## Account, budget, and API key setup

Everything so far ran entirely offline with fake tools and a fake brain. This section is
where you connect this course to a real model provider. It is the one-time setup that
Chapters 2, 3, 4, 6, 7, and the capstone all reuse, so it is worth doing carefully now.

This section assumes you have not necessarily used a terminal, `pip`, or environment
variables before. If you already have, skim it and move on; if you have not, each step below
says what it is and why it matters, not just the command to run.

### Step 1: Choose a provider

You have two options: Anthropic (`console.anthropic.com`) or OpenAI
(`platform.openai.com`). Either works throughout this entire course. The choice gets stored
in one environment variable (`LLM_PROVIDER`), and nothing else in the curriculum needs to
know which one you picked. If you do not already have a preference, either is a fine default;
pick whichever you are more likely to use professionally.

### Step 2: Create an account

Sign up at whichever console you chose. New accounts sometimes receive a small trial credit;
do not count on a specific amount, since offers change, but it is common to get a small
amount of free credit to start with.

### Step 3: Set a spend limit before you generate a key

Do this before step 4, not after. A spend limit is a hard ceiling on how much you can be
charged, independent of anything your code does. It is the single most important step in this
whole section, given that Chapters 1 and 2 are literally about teaching you to detect a
runaway agent loop. A limit means a bug cannot turn into a surprise bill while you are still
learning to catch bugs like that.

On Anthropic: in the console, go to Settings, then Billing, add a payment method, and set a
spending limit there. If that exact menu label has moved by the time you read this (console
UIs change), look under Billing/Usage for a spend or budget control, or simply leave
auto-reload disabled on prepaid credits, which caps your maximum possible spend at whatever
credit balance you have purchased, with no extra menu-hunting required.

On OpenAI: in the platform dashboard, go to your project's Limits page and set a monthly
budget there. Same caveat: if the exact path has moved, look for "Limits" or "Usage limits"
under your project or organization settings.

Recommended ceiling: \$15 to \$20 for the entire course. That is a comfortable margin. A full
pass through the course, including reasonable re-running of cells while learning,
realistically costs \$5 to \$15 on either provider's cheapest current-generation model (see the
cost note below). Console UIs change often, so confirm the exact current menu path against
your live dashboard rather than trusting this paragraph blindly.

### Step 4: Generate an API key

Once your spend limit is set, generate an API key from the same console.

### Step 5: Create a `.env` file

In the root of this repo, copy `.env.example` to a new file named `.env` and fill in:

```
LLM_PROVIDER=anthropic
ANTHROPIC_API_KEY=sk-ant-...
```

(or `LLM_PROVIDER=openai` with `OPENAI_API_KEY=...`, if that is what you picked). A `.env`
file is just a plain-text list of `NAME=value` pairs that gets loaded into your program's
environment at runtime. It is the standard way to keep secrets like API keys out of your
actual code and out of git.

### Step 6: Never commit `.env`

This repo's `.gitignore` already excludes `.env`, so `git add .` will never accidentally
stage it. Still worth knowing why: an API key committed to git history is effectively public
forever, even if you delete it in a later commit, because git remembers.

### Step 7: Install the SDKs

If you are working in this repo, `pip install -r requirements.txt` (from the root README's
setup instructions) already installs both `anthropic` and `openai`, regardless of which
provider you picked, so switching providers later never requires a fresh install.

### A note on cost

Real-API calls are the default across most of this course. Chapters 1, 2, 3, 4, 6, 7, and
the capstone all call a real model by default once this setup is done, falling back to a
mock/offline path (like everything above this section) only when no key is present. As of
this repo's last verification date (see `REFERENCES.md`):

Anthropic's Claude Haiku 4.5, this course's default model, is \$1/\$5 per million
input/output tokens. Anthropic's Claude Sonnet 5, the stronger tier some chapters use, is
\$2/\$10 per million tokens; Anthropic made this permanent on 2026-08-11, rather than the
price increase to \$3/\$15 that had originally been scheduled for September 1, 2026. OpenAI's
GPT-5.6 Luna, this course's default OpenAI model, has had inconsistently reported pricing
across sources during this repo's construction (note that the bare `"gpt-5.6"` alias
currently routes to a different, more expensive tier, so use the exact model ID
`gpt-5.6-luna`). Rather than assert a number here that is likely to be stale, check
`https://developers.openai.com/api/docs/pricing` for the current figure.

Per-token pricing on both platforms changes often; check the provider's live pricing page
before trusting any number written into a notebook months or years after it was built.

### No key, no budget? Use Ollama instead

If you would rather not create an account or spend anything at all, you can get real (if
smaller/weaker) model behavior for free by running a model locally via
[Ollama](https://ollama.com): install it, run `ollama pull llama3.2` (or any model you like)
from a terminal, and it serves an OpenAI-compatible API on `localhost:11434` with no account,
no key, and no spend limit needed at all. This course's `agentlib.llm_client` does not wire
this up automatically. It is a pointer for the curious, not a third code path this repo
maintains, but if you want to try it, point an OpenAI-compatible client at that local
endpoint instead of `platform.openai.com`.

In [12]:
from agentlib.llm_client import DEFAULT_MODELS, HAS_KEY, LLM_PROVIDER, STRONG_MODELS

print(f"LLM_PROVIDER = {LLM_PROVIDER!r}")
print(f"HAS_KEY      = {HAS_KEY}")

if HAS_KEY:
    print(f"Real API calls will use: {DEFAULT_MODELS[LLM_PROVIDER]} (default) "
          f"/ {STRONG_MODELS[LLM_PROVIDER]} (stronger tier, used by later chapters)")
else:
    print("No API key found -- this notebook will fall back to the deterministic mock brain "
          "built above. Complete the setup section above (or use Ollama) to try the real path.")


LLM_PROVIDER = 'anthropic'
HAS_KEY      = False
No API key found -- this notebook will fall back to the deterministic mock brain built above. Complete the setup section above (or use Ollama) to try the real path.


## Section 1: Definitions

### Agent vs. Chatbot vs. Workflow

An AI agent is software where the model decides the next step at runtime. Contrast this with
a chatbot, where control alternates strictly between human and model, and a workflow, where a
developer hardcodes every step in advance.

**Real-world agents:**

- **Devin** (Cognition, 2024): given a GitHub issue, it reads the codebase, writes code,
  runs tests, and opens a pull request. No human tells it which file to edit next; it
  decides based on test output.
- **Anthropic computer use** (2024): the model controls a mouse and keyboard on a real
  desktop, choosing which application to open and what to click based on screenshots it
  receives after each action.
- **Harvey** (2023): a legal AI agent that reads case filings, identifies relevant
  precedents, and drafts contract clauses. The attorney reviews the output, but the agent
  chose which precedents to pull and in what order.

| | Chatbot | Workflow | Agent |
|---|---|---|---|
| Control flow | Fixed: user asks, model replies, repeat | Fixed, predetermined sequence of steps written by a developer | Dynamic: the model decides what to do next at each step |
| Tool use | None, or a single hardcoded call | Deterministic calls to specific tools in a fixed order | The model chooses which tool(s) to call and when, based on what it observes |
| Autonomy | None: always waits for the next human turn | None: the sequence never deviates, regardless of what happens | Can take several actions in a row with no human turn in between |
| Typical failure mode | Cannot remember earlier turns; cannot take action | Breaks the moment reality does not match the checklist | Can loop forever or wander off-task without proper guardrails |

### The ReAct pattern

Most agent loops follow ReAct: reason, then act (Yao et al., 2022). The model interleaves
three things in a loop. First, a **thought**: the model reasons about what to do next.
Then an **action**: it picks a tool and produces structured input for it. Finally, an
**observation**: your code runs the tool and feeds the result back into the model's context.
The loop repeats until the model decides it has enough information to answer.

**Real-world ReAct implementations:**

- **ChatGPT with tools** (OpenAI, 2023): when a user asks "what is the weather in Tokyo,"
  the model decides to call a weather API, reads the JSON response, then writes a
  natural-language answer. Each tool call is a Thought $\rightarrow$ Action $\rightarrow$ Observation cycle.
- **Perplexity AI** (2023): given a research question, the model issues multiple search
  queries, reads the returned snippets, and synthesizes an answer with citations. It
  decides how many searches to run based on what each one returns.
- **GitHub Copilot Workspace** (2024): decomposes a coding task into steps, executes
  each step, checks the result, and revises if tests fail.

### When NOT to use an agent

Would you actually hire someone for this role, or would a simple checklist, or a vending
machine, do the job? Five concrete reasons to skip the agent:

1. **The task is fully deterministic.** Same input always produces the same correct output
   via fixed logic. Example: ETL pipeline transforming CSV rows into database records.
2. **A single LLM call suffices.** No tool use, no multi-step reasoning. Example:
   classifying customer feedback as positive/negative/neutral.
3. **High-stakes, irreversible actions at scale.** The blast radius of an autonomous
   mistake is too large. Example: a system that auto-approves medical prescriptions
   without pharmacist review.
4. **Latency budget is too tight.** Each agent step adds round-trip time. Example:
   autocomplete suggestions that must appear within 100ms.
5. **You cannot define success.** If you cannot write a test for "did the agent do the
   right thing," you cannot tell whether it is working. Example: "make the customer
   feel heard" with no measurable proxy.

Chapter 8 covers this judgment call in much more depth; this is the seed of it.

### Core vocabulary

**Token.** A chunk of text, often a sub-word piece, that is the model's basic unit of input
and output. "Unhappiness" might be three tokens: "un", "happiness", and a trailing space.
The exact split depends on the tokenizer.

**Context window.** The maximum number of tokens a model can attend to at once, covering
the system prompt, conversation history, and everything generated so far. Claude Haiku 4.5
has a 200k-token context window. GPT-5.6 Luna has a 128k-token window.

**Tool calling (function calling).** The model produces a structured request, such as
calling `calculator` with `{"expression": "12 * 7"}`, instead of free text; your code is
what actually executes it.

**Grounding.** Anchoring a model's output in specific, checkable source material (a
document, a tool result) rather than relying purely on what it learned during training.

**Hallucination.** The model confidently producing content that is false, unsupported, or
invented. Two real cases: in *Mata v. Avianca* (2023), a lawyer submitted a brief citing
six court decisions that did not exist, all generated by ChatGPT. In *Moffatt v. Air
Canada* (2024), Air Canada's chatbot invented a bereavement fare policy that the airline
had never offered, and the airline was held liable for the chatbot's fabrication.
Grounding reduces hallucination risk; it does not eliminate it (Chapter 3 shows exactly
why, hands-on).

## Section 2: Concept Explanation

### The ReAct loop, step by step

```
                +------------------+
                |   User task      |
                +--------+---------+
                         |
                         v
                +------------------+
           +--->|  THOUGHT         |
           |    |  (model reasons) |
           |    +--------+---------+
           |             |
           |             v
           |    +------------------+
           |    |  ACTION          |
           |    |  (pick a tool)   |-----> "final_answer" ----> DONE
           |    +--------+---------+
           |             |
           |             v
           |    +------------------+
           |    |  OBSERVATION     |
           |    |  (tool result)   |
           |    +--------+---------+
           |             |
           +-------------+
                loop
```

At each iteration, the model receives the full message history: the original task plus every
observation collected so far. It uses that history to decide the next action. The loop
terminates when the model emits `final_answer` instead of a tool name, or when an iteration
budget runs out.

### Why LLMs are stateless

A language model has no memory between calls. Every invocation starts fresh. What looks like
"memory" is your code concatenating previous messages into the next request. If you forget
to append a turn, the model genuinely does not know it happened. Section 4 (Build It
Yourself) makes this concrete: you will build the threading yourself and watch it break
when you skip a step.

### Why model output is untrusted input

The model's output is a string. It might contain valid JSON for a tool call. It might
contain Python code. It might contain `__import__('os').system('rm -rf /')`. Treating
model output the same way you treat user input from a web form is the right default: parse
it, validate it, reject anything unexpected. The calculator tool in Section 3 demonstrates
this by parsing expressions through Python's `ast` module instead of calling `eval()`
directly.

### Trade-offs to watch

**Iteration budget.** A higher `max_iterations` lets the agent attempt more steps, which
helps on complex tasks. But each step costs tokens (and money), and a stuck agent with a
high budget burns through that budget before stopping. The Break It section below
demonstrates this failure mode directly.

**Memory depth.** Including more conversation history gives the model better recall of
earlier context. But each prior turn costs input tokens, and very long histories can push
the model past its context window. Chapter 5 measures these costs precisely.

## Section 3: Example Code Segments

### The calculator tool

Every employee needs tools to do their job. We will give ours two: a calculator and a search
system. Both are deterministic and fully offline, so this chapter's output never depends on
network state or model randomness.

The calculator deliberately does not use Python's `eval()` on the raw string a model
produces, because that would let arbitrary code execution slip in through model output.
Instead it parses the expression into an AST and only allows numbers and basic arithmetic
operators.

In [2]:
import ast
import operator

# Map each AST node type to the Python operator it represents.
# Only arithmetic ops are allowed; anything else (function calls,
# attribute access, imports) gets rejected by _eval_node.
_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _eval_node(node):
    # Base case: a numeric literal (int or float).
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    # Recursive case: a binary operation like 3 + 4.
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    # Recursive case: a unary operation like -5.
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_eval_node(node.operand))
    # Anything else (names, calls, imports) is rejected outright.
    raise ValueError(f"Unsupported expression element: {ast.dump(node)}")


def calculator_tool(expression: str) -> dict:
    '''The 'calculator' resource: a restricted-eval arithmetic tool. Only numbers and
    +, -, *, /, ** are allowed -- anything else (names, calls, imports, ...) is rejected
    rather than executed, since this tool's input may come from model output.'''
    try:
        # Parse the string into an AST without executing it.
        tree = ast.parse(expression, mode="eval")
        # Walk the AST recursively, computing only allowed operations.
        result = _eval_node(tree.body)
        return {"status": "ok", "result": result}
    except Exception as exc:
        # Return the error as data, not an exception, so the agent loop
        # can show it as an observation rather than crashing.
        return {"status": "error", "error": str(exc)}


# Sanity checks: valid arithmetic, compound expression, and a blocked import attempt.
print(calculator_tool("12 * 7"))
print(calculator_tool("(3 + 4) * 2 - 1"))
print(calculator_tool("__import__('os').listdir()"))

{'status': 'ok', 'result': 84}
{'status': 'ok', 'result': 13}
{'status': 'error', 'error': "Unsupported expression element: Call(func=Attribute(value=Call(func=Name(id='__import__', ctx=Load()), args=[Constant(value='os')], keywords=[]), attr='listdir', ctx=Load()), args=[], keywords=[])"}


### The mock search tool

A small, deterministic, canned-fact lookup. A real search tool hits a live index; this one
hits a fixed dictionary so the rest of this chapter's output never depends on the internet
being up.

In [3]:
# Three canned facts, keyed by topic. The search tool matches
# against these keys so every run produces identical output.
_MOCK_FACTS = {
    "anthropic founder": (
        "Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with "
        "several colleagues who had previously worked at OpenAI."
    ),
    "react pattern": (
        "ReAct interleaves reasoning traces with actions, letting a model plan and use "
        "tools within the same loop (Yao et al., 2022)."
    ),
    "context window": (
        "A context window is the maximum number of tokens a model can attend to at once, "
        "including the prompt, conversation history, and its own output so far."
    ),
}


def mock_search_tool(query: str) -> dict:
    '''The 'search' resource: a small, deterministic, canned-fact lookup.'''
    q = query.lower()
    # Check each canned key; match if the key appears in the query
    # or any word from the key appears in the query.
    for key, fact in _MOCK_FACTS.items():
        if key in q or any(word in q for word in key.split()):
            return {"status": "ok", "result": fact}
    # No match found: return a not_found status so the agent loop
    # can report it as an observation without crashing.
    return {"status": "not_found", "result": f"No canned result for query: {query!r}"}


# One hit, one miss: verify both paths work.
print(mock_search_tool("who founded anthropic?"))
print(mock_search_tool("weather in tokyo"))

{'status': 'ok', 'result': 'Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.'}
{'status': 'not_found', 'result': "No canned result for query: 'weather in tokyo'"}


### The fake brain

Before wiring up a real model call (that comes later), we need something to play the role
of "the model deciding what to do." `fake_llm_brain()` below is a small rule-based function
that looks at what has happened so far and picks the next action via keyword matching. It
exists so the ReAct loop's mechanics can be taught and demonstrated for free, fully offline,
with fully deterministic output. It approximates what a real model does; it is not doing
what a real model does.

In [4]:
def fake_llm_brain(messages: list) -> dict:
    '''Rule-based stand-in for a real LLM 'brain' -- NOT reasoning, just deterministic
    keyword/state matching. Returns one of:
      {"action": "calculator",  "action_input": "<expression>"}
      {"action": "mock_search", "action_input": "<query>"}
      {"action": "final_answer","action_input": "<text>"}
    '''
    task_text = messages[0]["content"].lower()
    # Collect all observations the loop has appended so far.
    observations = [m for m in messages if m["role"] == "observation"]

    # Detect whether the task contains arithmetic.
    has_math = bool(re.search(r"\d+\s*[+\-*/]\s*\d+", task_text))
    # Detect whether the task asks about Anthropic's founding.
    wants_search = "anthropic" in task_text and any(
        w in task_text for w in ["who", "found", "search"]
    )

    # Track which tools have already been called.
    did_math = any(o["tool"] == "calculator" for o in observations)
    did_search = any(o["tool"] == "mock_search" for o in observations)

    # Dispatch: call calculator first if math is needed, then search.
    if has_math and not did_math:
        expr = re.search(r"[\d.\s+\-*/()]{3,}", task_text).group().strip()
        return {"action": "calculator", "action_input": expr}
    if wants_search and not did_search:
        return {"action": "mock_search", "action_input": "anthropic founder"}

    # All needed tools have been called; assemble a final answer
    # from their collected results.
    parts = [str(o["content"].get("result", o["content"])) for o in observations]
    answer = " ".join(parts) if parts else "I don't have enough information to answer."
    return {"action": "final_answer", "action_input": answer}

### The real brain: provider-agnostic LLM client

`RealLLMBrain` is backed by a real model call through `agentlib.llm_client`, with proper
tool-use schemas for the calculator and search resources. It matches `fake_llm_brain`'s
`brain(messages) -> {"action": ..., "action_input": ...}` interface, so `run_agent()` can
use either brain interchangeably.

Building against one vendor's SDK everywhere is a common early mistake that creates painful
lock-in later. `agentlib.llm_client.call_model()` reads `LLM_PROVIDER` once and normalizes
both providers' responses into one common shape, so the rest of this course never has to
branch on which provider is active.

The interesting problem in `RealLLMBrain` is an impedance mismatch worth understanding.
`run_agent()` is sequential: it asks for ONE action, executes it, and comes back with one
observation. But a model is not obliged to cooperate with that. Asked "how many days in 12
weeks, and when was the company founded," it often returns TWO tool calls in a single
turn (parallel tool use). The provider requires that every tool call gets a matching result;
reply with only one and the request is rejected. So `RealLLMBrain` queues the extra calls
and serves them to the loop one at a time.

In [13]:
from agentlib import llm_client

CALCULATOR_TOOL_SCHEMA = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression (+, -, *, /, **). Input must be "
                    "a plain arithmetic expression string, e.g. '12 * 7'.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"],
    },
}

MOCK_SEARCH_TOOL_SCHEMA = {
    "name": "mock_search",
    "description": "Search a small internal knowledge base for a fact. Input is a short "
                    "query string.",
    "input_schema": {
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
}

AGENT_SYSTEM_PROMPT = (
    "You are a helpful employee with access to a calculator and a search tool. Use them "
    "when needed to answer the user's request accurately, then give a final answer in "
    "plain text with no further tool calls."
)


# Which input field each tool's schema declares, so the brain can pull the argument out of a
# tool call without guessing from the tool's name.
TOOL_INPUT_KEY = {"calculator": "expression", "mock_search": "query"}


class RealLLMBrain:
    '''Stateful wrapper around agentlib.llm_client.call_model() that maintains provider-native
    tool-call turn structure across a single agent run, while matching fake_llm_brain's simple
    brain(messages) -> action-dict interface for run_agent().

    The interesting problem here is an impedance mismatch, and it is worth understanding
    because you will hit it in any real agent loop. run_agent() is sequential: it asks for ONE
    action, executes it, and comes back with one observation. A model is not obliged to
    cooperate with that. Asked something like "how many days in 12 weeks, and when was the
    company founded", it will often return TWO tool calls in a single turn -- parallel tool
    use -- because both are answerable at once.

    That produces one assistant turn carrying two `tool_use` blocks, and the provider's rule is
    that every one of them must be answered by a matching `tool_result`. Reply with a single
    result and the request is rejected outright. It is an intermittent failure, because
    whether the model parallelises depends on the question.

    So this class queues the extra calls and serves them to the loop one at a time, buffering
    each observation until the queue drains, then emitting all the results together via
    format_tool_results() -- which batches them the way the active provider expects.
    '''

    def __init__(self, model: str | None = None):
        self.model = model or llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER]
        self.provider_messages: list[dict] = []
        self._pending_tool_calls: list = []      # calls from a parallel turn, not yet served
        self._buffered_results: list = []        # (ToolCall, observation) awaiting a flush
        self._last_tool_call = None

    def _serve(self, tool_call) -> dict:
        """Hand one tool call to the agent loop in its expected action-dict shape."""
        self._last_tool_call = tool_call
        key = TOOL_INPUT_KEY.get(tool_call.name, "query")
        return {"action": tool_call.name, "action_input": tool_call.input.get(key, "")}

    def __call__(self, messages: list) -> dict:
        if not self.provider_messages:
            self.provider_messages.append({"role": "user", "content": messages[0]["content"]})
        else:
            # The loop just executed whatever we served last; hold onto the result rather than
            # appending it, so parallel results can be emitted as one correctly-shaped turn.
            self._buffered_results.append((self._last_tool_call, json.dumps(messages[-1]["content"])))

        # Still working through a parallel turn: serve the next call without consulting the
        # model, since it already told us what it wanted.
        if self._pending_tool_calls:
            return self._serve(self._pending_tool_calls.pop(0))

        # Queue drained -- every tool_use from the last assistant turn now has an answer.
        if self._buffered_results:
            self.provider_messages.extend(llm_client.format_tool_results(self._buffered_results))
            self._buffered_results = []

        response = llm_client.call_model(
            messages=self.provider_messages,
            system=AGENT_SYSTEM_PROMPT,
            tools=[CALCULATOR_TOOL_SCHEMA, MOCK_SEARCH_TOOL_SCHEMA],
            model=self.model,
        )

        if response.tool_calls:
            # Append the assistant turn as soon as it arrives, exactly once. Deferring it to
            # the next invocation makes it far too easy to append it twice, or not at all.
            self.provider_messages.append(llm_client.format_assistant_tool_call(response))
            self._pending_tool_calls = list(response.tool_calls[1:])
            return self._serve(response.tool_calls[0])

        return {"action": "final_answer", "action_input": response.text}


## Section 4: Build It Yourself

### Task 1: The ReAct loop (`run_agent`)

This is the actual Thought $\rightarrow$ Action $\rightarrow$ Observation loop, and it is yours to write.
Ask the brain what to do, run the tool it picked, feed the result back in, repeat until it
says it is done (or you hit an iteration cap).

The one part worth thinking about before you type is the "feed the result back in" step.
The brain is stateless; the only thing it knows about what has already happened is what it
can read in `messages`. An observation that never gets appended is an observation the brain
never sees, so it will keep asking for the same thing forever.

Every graded cell in this course ends with `your_function = check("task-id", your_function)`.
That call runs a suite of assertions against what you wrote and, if any fail, prints exactly
which ones and why; it hands your function straight back, so the rebinding is a no-op and
later cells just use the name as normal. Run `python grade.py` at any point to see where you
stand.

In [ ]:
TOOLS = {
    "calculator": calculator_tool,
    "mock_search": mock_search_tool,
}


def run_agent(task: str, brain, tools: dict, max_iterations: int = 6, verbose: bool = True) -> str:
    '''Run the ReAct loop until the brain gives a final answer or the budget runs out.

    brain(messages) -> {"action": ..., "action_input": ...}, where action is either
    "final_answer" or a key of `tools`. Seed `messages` with the task as a user turn:

        {"role": "user", "content": task}

    and append every tool result back onto it as:

        {"role": "observation", "tool": <action>, "content": <what the tool returned>}

    Return the final answer's action_input. If the brain names a tool that doesn't exist,
    return a message saying so rather than raising; if the budget runs out, return a message
    saying that instead.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


run_agent = check("ch01-react-loop", run_agent)

In [6]:
result = run_agent("What is 12 * 7? Also, who founded Anthropic?", fake_llm_brain, TOOLS)
print("\n=== RESULT ===")
print(result)

[step 1] Thought -> Action: calculator('12 * 7')
[step 1] Observation: {'status': 'ok', 'result': 84}
[step 2] Thought -> Action: mock_search('anthropic founder')
[step 2] Observation: {'status': 'ok', 'result': 'Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.'}
[step 3] Thought -> Action: final_answer('84 Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.')
[step 3] Final answer: 84 Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.

=== RESULT ===
84 Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.


### Task 2: Conversation memory (`chat_turn`)

Separate from the ReAct tool loop above, a real agent also has to manage conversation
memory across turns: whether earlier turns get threaded back into the prompt at all. There is
no separate memory mechanism unless you build one; "memory" is conversation history being
concatenated back into context on every call, which means it is your job to put it there.

You write the threading. The brain below is given, and it can only recall what it can read in
the history it is handed.

In [ ]:
def fake_llm_brain_chat(history: list, user_input: str) -> str:
    '''Toy conversational stand-in for this memory demo only -- separate from
    fake_llm_brain's ReAct loop above, since conversational memory and within-task tool
    memory are two different things a real agent has to manage. It recalls a name only if
    the name is somewhere in the history it is given.'''
    context_text = " ".join(m["content"] for m in history)
    if "what is my name" in user_input.lower():
        match = re.search(r"my name is (\w+)", context_text, re.IGNORECASE)
        if match:
            return f"Your name is {match.group(1)}."
        return "I don't know your name -- you haven't told me, or I'm not remembering earlier turns."
    return "Noted."


def chat_turn(history: list, user_input: str, brain) -> str:
    '''Run one conversational turn and thread it into `history`.

    Call brain(history, user_input) -- the brain sees the conversation as it stood BEFORE
    this turn, plus the new input. Then record both halves of the exchange onto `history`
    in place, as:

        {"role": "user", "content": user_input}
        {"role": "assistant", "content": <the reply>}

    Return the reply.

    Both halves. Recording only the agent's own replies is the classic half-implementation:
    everything the user actually told you disappears.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


chat_turn = check("ch01-memory", chat_turn)

In [8]:
history = []

turn1 = "My name is Jack."
print("User:", turn1)
print("Agent:", chat_turn(history, turn1, fake_llm_brain_chat))

turn2 = "What is my name?"
print("\nUser:", turn2)
# Memory OFF is not a separate mode -- it is simply not passing the history along.
print("Agent (memory OFF):", fake_llm_brain_chat([], turn2))
print("Agent (memory ON): ", fake_llm_brain_chat(history, turn2))

User: My name is Jack.
Agent: Noted.

User: What is my name?
Agent (memory OFF): I don't know your name -- you haven't told me, or I'm not remembering earlier turns.
Agent (memory ON):  Your name is Jack.


With memory off, the earlier turn never gets threaded back into what the model sees. It is
answering from nothing, the same way a new employee would if every request landed on their
desk with zero context about the conversation so far. In a real prompt, "memory" is just
conversation history being concatenated back into context on every call; there is no separate
memory mechanism unless you build one (Chapter 4 covers caching and staleness in that
history).

## Section 5: Playground

These experiments use the functions you built in Section 4. Tweak the parameters, re-run
each cell, and observe how behavior changes.

### The toggle: real API vs. mock

When a key is present, every run below uses a real model; when it is not (as in CI, or for
a learner without budget), it transparently falls back to the deterministic mock brain.
Same `run_agent()` function, same interface, zero code changes required either way.

In [14]:
brain = RealLLMBrain() if HAS_KEY else fake_llm_brain
print(f"Using: {'RealLLMBrain (real API call)' if HAS_KEY else 'fake_llm_brain (mock, no key present)'}")

final = run_agent("What is 12 * 7? Also, who founded Anthropic?", brain, TOOLS)
print("\n=== RESULT ===")
print(final)


Using: fake_llm_brain (mock, no key present)
[step 1] Thought -> Action: calculator('12 * 7')
[step 1] Observation: {'status': 'ok', 'result': 84}
[step 2] Thought -> Action: mock_search('anthropic founder')
[step 2] Observation: {'status': 'ok', 'result': 'Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.'}
[step 3] Thought -> Action: final_answer('84 Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.')
[step 3] Final answer: 84 Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.

=== RESULT ===
84 Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.


### Experiment 1: Iteration budget

What happens when `max_iterations` is too low for the task? Too high for a stuck agent?

In [ ]:
# Try each value: 1, 3, 6, 20. Watch how the agent
# behaves differently with each budget.
for budget in [1, 3, 6, 20]:
    print(f"\n{'='*60}")
    print(f"max_iterations = {budget}")
    print(f"{'='*60}")
    result = run_agent(
        "What is 12 * 7? Also, who founded Anthropic?",
        fake_llm_brain,
        TOOLS,
        max_iterations=budget,
        verbose=True,
    )
    print(f"Result: {result[:80]}...")

### Experiment 2: Tool availability

What happens when the agent needs a tool that is not available?

In [ ]:
# Four configurations: both tools, calculator only, search only, neither.
tool_configs = {
    "both": {"calculator": calculator_tool, "mock_search": mock_search_tool},
    "calc_only": {"calculator": calculator_tool},
    "search_only": {"mock_search": mock_search_tool},
    "none": {},
}

for label, tool_set in tool_configs.items():
    print(f"\n{'='*60}")
    print(f"Tools available: {label} ({list(tool_set.keys())})")
    print(f"{'='*60}")
    result = run_agent(
        "What is 12 * 7? Also, who founded Anthropic?",
        fake_llm_brain,
        tool_set,
        max_iterations=6,
        verbose=True,
    )
    print(f"Result: {result[:80]}")

### Experiment 3: Conversation history depth

Does the agent remember earlier turns? What happens when you clear history mid-conversation?

In [ ]:
# Build up 3 turns of history, then ask the recall question
# at different history depths.
full_history = []
chat_turn(full_history, "My name is Alice.", fake_llm_brain_chat)
chat_turn(full_history, "I work at Acme Corp.", fake_llm_brain_chat)
chat_turn(full_history, "My favorite color is blue.", fake_llm_brain_chat)

print("Full history (3 prior turns):")
print(f"  Q: 'What is my name?' -> {fake_llm_brain_chat(full_history, 'What is my name?')}")

# Only the most recent turn (no name mentioned there).
print("\nLast turn only:")
print(f"  Q: 'What is my name?' -> {fake_llm_brain_chat(full_history[-2:], 'What is my name?')}")

# Empty history.
print("\nNo history:")
print(f"  Q: 'What is my name?' -> {fake_llm_brain_chat([], 'What is my name?')}")

## Section 6: Break It

### The employee who never escalates

Here is the failure mode this chapter is really about: an employee assigned a task who gets
stuck and just keeps trying the same thing, forever, without ever stopping to say "this is
not working, I need to ask for help." We will build a "bait tool" that always reports it is
not done yet, and a brain with no escalation logic, and watch what happens.

**Production impact:** A runaway loop like this burns tokens on every iteration. With no
spend limit and no iteration cap, a single stuck agent can accumulate hundreds of dollars
in API charges overnight. The spend limit you set in the Account Setup section above is
your first line of defense; the duplicate-observation guard you build below is the second.

In [9]:
def bait_tool(_input):
    '''Always reports 'still working on it', regardless of input -- models a system (or
    an employee) that never actually finishes and never escalates.'''
    return {"status": "partial", "detail": "Still processing your request, check back."}


def looping_brain(messages):
    '''No escalation logic: if the last observation says the task isn't done, ask again.
    This is the bug.'''
    observations = [m for m in messages if m["role"] == "observation"]
    if not observations or observations[-1]["content"].get("status") == "partial":
        return {"action": "bait_tool", "action_input": "any"}
    return {"action": "final_answer", "action_input": "done"}


print("--- Demonstrating the bug: no stop condition, no escalation ---\n")
buggy_result = run_agent(
    "Generate my quarterly report.",
    looping_brain,
    tools={"bait_tool": bait_tool},
    max_iterations=20,  # capped ONLY so this notebook doesn't hang -- see note below
    verbose=True,
)
print("\n=== RESULT ===")
print(buggy_result)


--- Demonstrating the bug: no stop condition, no escalation ---

[step 1] Thought -> Action: bait_tool('any')
[step 1] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 2] Thought -> Action: bait_tool('any')
[step 2] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 3] Thought -> Action: bait_tool('any')
[step 3] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 4] Thought -> Action: bait_tool('any')
[step 4] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 5] Thought -> Action: bait_tool('any')
[step 5] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 6] Thought -> Action: bait_tool('any')
[step 6] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 7] Thought -> Action: bait_tool('any')
[step 7] Observation

Twenty identical "still processing" observations in a row, and the loop only stopped
because we artificially capped `max_iterations` at 20 purely so this notebook would not hang
forever. A real production agent has no such cap by default, which is exactly how a runaway
agent burns unbounded tokens (and money) on a bug like this.

### Your fix: duplicate-observation detection

The fix is yours to write: if the exact same observation fires twice in a row with no
progress, stop and escalate instead of continuing. That is the equivalent of an employee
saying "I have asked twice and gotten the same non-answer both times, I need to flag this"
instead of asking a third, fourth, fifth time.

Note the exact rule: twice in a row. Not "an observation I have seen at some point." An
agent that revisits an earlier state after doing real work in between is making progress,
and a guard that remembers every observation forever would kill it for no reason.

**Hint 1:** What should you compare against? Think about which previous observation matters.

**Hint 2:** Only track the PREVIOUS observation, not the full history. Compare the current
result to the one immediately before it.

**Interview follow-up:** "How would you detect and stop a looping agent in production?"

In [ ]:
def run_agent_with_guard(task: str, brain, tools: dict, max_iterations: int = 50, verbose: bool = True) -> str:
    '''Same loop as run_agent(), but escalates instead of repeating once the exact same
    observation fires twice IN A ROW.

    Everything run_agent() does still applies: seed messages with the task, run the tool the
    brain names, thread each observation back. The addition is a comparison against the
    previous observation only -- if this one is identical, stop and return a message
    explaining that you are escalating (the word "escalating" should appear in it) instead
    of appending the observation and looping again.

    Tool results are dicts, so compare them by value, not by identity.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


run_agent_with_guard = check("ch01-dup-guard", run_agent_with_guard)

In [11]:
print("--- Demonstrating the fix: duplicate-observation detection ---\n")
fixed_result = run_agent_with_guard(
    "Generate my quarterly report.",
    looping_brain,
    tools={"bait_tool": bait_tool},
    max_iterations=50,
    verbose=True,
)
print("\n=== RESULT ===")
print(fixed_result)

--- Demonstrating the fix: duplicate-observation detection ---

[step 1] Thought -> Action: bait_tool('any')
[step 1] Observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
[step 2] Thought -> Action: bait_tool('any')
[step 2] Escalating after step 2: received the identical observation twice in a row ({'status': 'partial', 'detail': 'Still processing your request, check back.'}) with no progress. A real employee would ask for help here instead of repeating the same request forever.

=== RESULT ===
Escalating after step 2: received the identical observation twice in a row ({'status': 'partial', 'detail': 'Still processing your request, check back.'}) with no progress. A real employee would ask for help here instead of repeating the same request forever.


Before: 20 iterations of identical spam, stopped only by an artificial cap. After: 2 steps,
then a clear escalation message. Same buggy brain, same bait tool; the only difference is
the guard.

## Section 7: Interview Q&A

### Key takeaways

- An agent differs from a chatbot or a workflow by dynamic control flow: the model decides
  what to do next, not just what to say next.
- ReAct (Thought $\rightarrow$ Action $\rightarrow$ Observation) is the loop shape underneath most agents,
  including the one you just built.
- An agent with no stop condition and no escalation logic can loop forever, burning
  unbounded cost. Duplicate-observation detection is one concrete, cheap guard against it
  (Chapter 2 builds a proper max-iteration guard and a harder-to-catch cycle variant).
- Grounding reduces hallucination risk, but it does not eliminate it.
- Building against a provider-agnostic client rather than one vendor's SDK directly is a
  real production practice, not just a teaching convenience.

### Model answers

Five questions that map directly to real interview phrasings. Attempt them from memory
before revealing the model answer.

1. **"Your agent keeps looping forever. How do you detect and stop it?"**
   Compare consecutive observations. If two adjacent observations are identical, the agent
   is stuck. Stop the loop and escalate (return an error, alert a human, or fall back to a
   simpler strategy). Also enforce a hard iteration cap as a backstop, and set a spend
   limit on the provider account so a missed bug cannot run up an unbounded bill.

2. **"A stakeholder says we do not need an agent. When are they right?"**
   When the task is fully deterministic (a fixed pipeline), when a single LLM call
   suffices (classification, summarization), when the action is high-stakes and
   irreversible at scale (auto-approving financial transactions), when latency
   requirements rule out multi-step reasoning, or when you cannot define a measurable
   success criterion for the agent's output.

3. **"Explain chatbot vs. workflow vs. agent to a non-technical colleague."**
   A chatbot is a help desk: one question, one answer, repeat. A workflow is a checklist:
   step 1 through step N, no deviation. An agent is an employee given a goal who decides
   which resources to use and in what order, taking multiple steps on their own before
   reporting back.

4. **"What is the difference between context window and memory?"**
   The context window is a hard token limit on how much text the model can see at once.
   "Memory" is your code appending prior conversation turns into that window. The model
   itself stores nothing between calls. If you do not append a turn, the model does not
   know it happened, regardless of how large the context window is.

5. **"Why can a RAG-grounded model still hallucinate?"**
   Grounding provides the right source material in context. The model can still ignore it,
   misinterpret it, or combine pieces of it incorrectly. The retrieval step can also fail:
   the right document might not be retrieved, or it might be retrieved but buried at a low
   rank. Chapter 3 covers each of these failure surfaces in detail.

### Vocabulary flashcards (optional, interactive)

Skippable: if you are not running this in an interactive terminal/Jupyter session (e.g. this
cell is being executed headlessly in CI), it will detect that and skip itself automatically.

In [15]:
FLASHCARDS = [
    ("Context window", "The maximum number of tokens (prompt + history + output-so-far) a model can attend to at once."),
    ("Tool / function calling", "A model producing a structured request to invoke an external function, rather than free text -- your code does the actual invoking."),
    ("Grounding", "Anchoring a model's output in specific, checkable source material rather than relying purely on parametric knowledge."),
    ("Hallucination", "A model confidently producing content that's false, unsupported, or invented."),
    ("ReAct", "Thought -> Action -> Observation: an agent pattern interleaving reasoning traces with tool actions (Yao et al., 2022)."),
    ("Token", "A chunk of text, often a sub-word piece, that's the model's basic unit of input and output."),
]


def flashcard_quiz(cards: list, interactive: bool = True) -> None:
    if not interactive:
        print("Quiz skipped (interactive=False).")
        return
    for term, definition in cards:
        try:
            input(f"Define: {term}\n> ")
        except Exception:
            # Covers both a plain EOFError (stdin closed) and Jupyter's
            # StdinNotImplementedError (headless kernel, e.g. this notebook running in CI) --
            # either way, there's no one there to type an answer, so skip gracefully.
            print(
                "\n(No interactive input available -- skipping the rest of the quiz. Run "
                "this cell in a real terminal or Jupyter session to try it for real.)"
            )
            return
        print(f"Reference definition: {definition}\n")
    print("Quiz complete.")


flashcard_quiz(FLASHCARDS, interactive=True)



(No interactive input available -- skipping the rest of the quiz. Run this cell in a real terminal or Jupyter session to try it for real.)


### Explain it to a non-technical PM

Write, in 3-5 plain-English sentences, how you would explain what an AI agent is to a
product manager who has never used one, without jargon (no "ReAct," "tool calling," "context
window"). The employee metaphor from this chapter's concept section is fair game if it
helps.

In [16]:
my_pm_explanation = '''
(Write your answer here.)
'''

print(my_pm_explanation)



(Write your answer here.)



### Written drill

Attempt these from memory before checking `solutions/ch01_fundamentals_answers.md`. These
map directly to real interview phrasings.

There is a slot below for each question. Write your answer into it, run the cell, then use
`drill.check(n)` to see your answer and the model answer side by side.

`check(n)` will not show you an answer until you have written one of your own. Once you have
read the model answer you can no longer find out what you actually knew. If you want it
anyway, `drill.reveal(n)` is there and makes no judgement.

Answers are read from `solutions/ch01_*_answers.md` at runtime, so nothing in this
notebook contains one.

In [17]:
from agentlib.self_check import drill as open_drill

drill = open_drill(1)
drill.questions()

Chapter 1 written drill — 5 questions

1. Your AI agent keeps looping forever. How would you detect and stop it?
2. A stakeholder says "we don't need an agent here, a simple script would do." When are they
   right?
3. Explain the difference between a chatbot, a workflow, and an agent to a non-technical
   colleague.
4. What's the difference between a model's context window and "memory" across a conversation?
5. Why can a RAG-grounded model still hallucinate?


In [18]:
# One slot per question. Replace the placeholder text, then run this cell.
# Anything under 25 words, or left as the placeholder, is not recorded.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

# Question 5
drill.attempt(5, '''
(Your answer here.)
''')

print()
drill.status()

  1. not recorded — it is still the placeholder
  2. not recorded — it is still the placeholder
  3. not recorded — it is still the placeholder
  4. not recorded — it is still the placeholder
  5. not recorded — it is still the placeholder

Chapter 1: 0/5 answered
  still open: [1, 2, 3, 4, 5]


In [19]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

Question 1 has no recorded answer yet.

  Your AI agent keeps looping forever. How would you detect and stop it?

Write one with attempt() first. Reading the model answer before you have committed to your own turns this into a reading exercise -- once you have seen it you can no longer find out what you actually knew.
(If you really want it anyway: reveal(1).)


## Section 8: References

1. Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2022).
   *ReAct: Synergizing Reasoning and Acting in Language Models.*
   [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)

2. Wei, J., Wang, X., Schuurmans, D., Bosma, M., Ichter, B., Xia, F., Chi, E., Le, Q.,
   & Zhou, D. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language
   Models.* [arXiv:2201.11903](https://arxiv.org/abs/2201.11903)

3. Anthropic API documentation: https://docs.anthropic.com

4. OpenAI API documentation: https://platform.openai.com/docs

5. *Mata v. Avianca* (S.D.N.Y. 2023). Case No. 22-cv-1461. Lawyer sanctioned for citing
   six fabricated court decisions generated by ChatGPT.

6. *Moffatt v. Air Canada* (CRT 2024). Chatbot invented a bereavement fare policy; airline
   held liable for the fabrication.

### Related chapters

- **Chapter 2** extends the single-agent ReAct loop to multi-agent teams with subagents.
- **Chapter 3** covers RAG and retrieval evaluation, including why grounded models still
  hallucinate.
- **Chapter 4** adds production reliability patterns (caching, retries, circuit breakers)
  to the tool-calling layer you built here.
- **Chapter 6** addresses the security implications of treating model output as untrusted
  input.

## Next: Chapter 2: Agent Control Flow

This chapter built one employee working alone. Chapter 2 turns that into a small team: Jack
(planner), Bob (worker), and Mike (critic). It introduces subagents: giving a teammate a
bounded task with a fresh, isolated context, and getting back a compressed summary instead
of their entire raw train of thought. It also covers a harder-to-catch failure mode than the
straight-line loop you just fixed: a genuine cycle, where two employees defer the same
decision back and forth without ever resolving it.